# Q2: Healthcare Access vs Health Outcomes

**Question:** Is there a relationship between healthcare access and health outcomes (mortality, recovery)?

**Sources:** `v_access_vs_outcomes`, `clean_health` (10K sample for scatter)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from notebooks._helpers import (
    get_connection, set_plot_style, save_fig, format_ci,
    TIER_COLORS,
)

set_plot_style()
con = get_connection()
print('Connected to DuckDB')

## 1. Load Data

In [ ]:
access_df = con.execute('SELECT * FROM v_access_vs_outcomes').fetchdf()
print(f'v_access_vs_outcomes: {len(access_df):,} rows')

# 10K sample for scatter plots
sample = con.execute("""
    SELECT healthcare_access, mortality_rate, recovery_rate,
           urbanization_tier, dalys
    FROM clean_health
    WHERE data_quality_flag = 'ok'
    USING SAMPLE 10000
""").fetchdf()
print(f'Scatter sample: {len(sample):,} rows')

## 2. Correlation Analysis

In [ ]:
# Pearson and Spearman on full aggregated data
pearson_r, pearson_p = stats.pearsonr(access_df['avg_access'], access_df['avg_mortality'])
spearman_r, spearman_p = stats.spearmanr(access_df['avg_access'], access_df['avg_mortality'])

print(f'Pearson  r = {pearson_r:.6f}  (p = {pearson_p:.2e})')
print(f'Spearman r = {spearman_r:.6f}  (p = {spearman_p:.2e})')

# OLS R-squared
slope, intercept, r_value, p_value, std_err = stats.linregress(
    sample['healthcare_access'], sample['mortality_rate']
)
r_squared = r_value ** 2
print(f'\nOLS Regression on 10K sample:')
print(f'  slope = {slope:.6f}, intercept = {intercept:.4f}')
print(f'  R-squared = {r_squared:.6f}')
print(f'  Interpretation: Access explains {r_squared*100:.4f}% of mortality variance')

## 3. Threshold Effects — Decile Analysis

In [ ]:
decile_df = con.execute("""
    WITH deciled AS (
        SELECT
            NTILE(10) OVER (ORDER BY healthcare_access) AS access_decile,
            healthcare_access,
            mortality_rate,
            recovery_rate
        FROM clean_health
        WHERE data_quality_flag = 'ok'
    )
    SELECT
        access_decile,
        AVG(healthcare_access) AS avg_access,
        AVG(mortality_rate) AS avg_mortality,
        AVG(recovery_rate) AS avg_recovery,
        COUNT(*) AS n
    FROM deciled
    GROUP BY access_decile
    ORDER BY access_decile
""").fetchdf()
decile_df

## 4. Visualization 1 — Scatter + Regression Line

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sample['healthcare_access'], sample['mortality_rate'],
           alpha=0.15, s=8, color='#3498db')

# Regression line
x_line = np.linspace(sample['healthcare_access'].min(),
                     sample['healthcare_access'].max(), 100)
y_line = slope * x_line + intercept
ax.plot(x_line, y_line, 'r-', linewidth=2,
        label=f'OLS: y = {slope:.4f}x + {intercept:.2f}\n$R^2$ = {r_squared:.6f}')

ax.set_xlabel('Healthcare Access (%)')
ax.set_ylabel('Mortality Rate (%)')
ax.set_title('Q2: Healthcare Access vs Mortality Rate (10K sample)')
ax.legend(loc='upper right')
save_fig(fig, 'q2_scatter_access_mortality')
plt.show()

## 5. Visualization 2 — Per-Group Correlation Histogram

In [ ]:
# Per-group correlations from the view
group_corrs = access_df['corr_access_mortality'].dropna()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(group_corrs, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='r = 0')
ax.axvline(group_corrs.mean(), color='orange', linestyle='-', linewidth=1.5,
           label=f'Mean r = {group_corrs.mean():.4f}')
ax.set_xlabel('Pearson r (access vs mortality)')
ax.set_ylabel('Number of subgroups')
ax.set_title('Q2: Distribution of Per-Group Correlations')
ax.legend()
save_fig(fig, 'q2_correlation_histogram')
plt.show()

## 6. Visualization 3 — Decile Step Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.step(decile_df['access_decile'], decile_df['avg_mortality'],
        where='mid', color='#e74c3c', linewidth=2, marker='o', markersize=6)
ax.set_xlabel('Healthcare Access Decile (1=lowest, 10=highest)')
ax.set_ylabel('Mean Mortality Rate (%)')
ax.set_title('Q2: Mean Mortality by Healthcare Access Decile')
ax.set_xticks(range(1, 11))

# Show how flat the line is
mortality_range = decile_df['avg_mortality'].max() - decile_df['avg_mortality'].min()
ax.annotate(f'Total range: {mortality_range:.4f}pp',
            xy=(5, decile_df['avg_mortality'].mean()),
            fontsize=11, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

save_fig(fig, 'q2_decile_step_chart')
plt.show()

## 7. Visualization 4 — Faceted Scatter by Tier

In [ ]:
tier_order = ['Rural', 'Peri-urban', 'Urban']
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, tier in zip(axes, tier_order):
    subset = sample[sample['urbanization_tier'] == tier]
    ax.scatter(subset['healthcare_access'], subset['mortality_rate'],
              alpha=0.15, s=8, color=TIER_COLORS[tier])
    # Per-tier regression
    if len(subset) > 10:
        s, i, r, p, se = stats.linregress(subset['healthcare_access'], subset['mortality_rate'])
        x_l = np.linspace(subset['healthcare_access'].min(), subset['healthcare_access'].max(), 50)
        ax.plot(x_l, s * x_l + i, 'k-', linewidth=1.5)
        ax.set_title(f'{tier}\n$R^2$ = {r**2:.6f}')
    else:
        ax.set_title(tier)
    ax.set_xlabel('Healthcare Access (%)')

axes[0].set_ylabel('Mortality Rate (%)')
fig.suptitle('Q2: Access vs Mortality by Urbanization Tier', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'q2_faceted_scatter_by_tier')
plt.show()

## 8. Key Finding

**There is NO relationship between healthcare access and health outcomes.**

- Pearson/Spearman r near zero across all groupings
- OLS R-squared < 0.001 — access explains virtually none of the mortality variance
- Decile analysis shows a flat line — no threshold effects
- This holds uniformly across all urbanization tiers

**The absence of a relationship IS the finding.** In real-world data, we would expect moderate positive correlations between access and recovery, and negative correlations between access and mortality. The complete absence of these expected relationships is a strong signal of synthetic data generation.

In [ ]:
con.close()
print('Q2 analysis complete.')